# TP – Préparation de données avec Spark : consommation d'énergie des bâtiments

**Objectif :** préparer le jeu de données *Building Energy Benchmarking* pour la prédiction de la consommation d'énergie des bâtiments non résidentiels (`SiteEnergyUse(kBtu)`), entièrement avec **PySpark**, sans pandas.

**Étapes :** session Spark → lecture du CSV → valeurs manquantes → nouvelles variables → conversion des nombres → filtrage → corrélations → fuites de données → redondances → encodage → sauvegarde Parquet.

Environnement : Google Colab, PySpark 4.x.

## 1. Session Spark

In [ ]:
!pip install -q pyspark

In [ ]:
import glob, os
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, StringType, NumericType

spark = SparkSession.builder.appName("BuildingEnergy").master("local[*]").getOrCreate()
spark

## 2. Lecture du CSV

Le fichier peut être déposé dans `/content` (panneau Fichiers de Colab, supprimé à la fin de la session) ou sur Google Drive (`drive.mount`).

In [ ]:
# Pour Google Drive : from google.colab import drive ; drive.mount('/content/drive')
fichiers = (glob.glob("/content/Building_Energy_Benchmarking*.csv")
            + glob.glob("/content/drive/MyDrive/**/Building_Energy_Benchmarking*.csv", recursive=True))
CHEMIN = os.environ.get("CHEMIN_CSV") or fichiers[0]
print(CHEMIN)

data = spark.read.csv(CHEMIN, header=True, inferSchema=True)
print(data.count(), "lignes,", len(data.columns), "colonnes")

## 3. Variables de travail

In [ ]:
features_variables = [
    'DataYear', 'BuildingName', 'BuildingType', 'TaxParcelIdentificationNumber', 'City', 'State',
    'ZipCode', 'Latitude', 'Longitude', 'Neighborhood', 'CouncilDistrictCode', 'YearBuilt',
    'NumberofFloors', 'NumberofBuildings', 'PropertyGFATotal', 'PropertyGFABuildings',
    'PropertyGFAParking', 'SelfReportGFATotal', 'SelfReportGFABuildings', 'SelfReportParking',
    'ENERGYSTARScore', 'SiteEUIWN(kBtu/sf)', 'SiteEUI(kBtu/sf)', 'SiteEnergyUse(kBtu)',
    'SiteEnergyUseWN(kBtu)', 'SourceEUIWN(kBtu/sf)', 'SourceEUI(kBtu/sf)', 'EPAPropertyType',
    'LargestPropertyUseType', 'LargestPropertyUseTypeGFA', 'SecondLargestPropertyUseType',
    'SecondLargestPropertyUseTypeGFA', 'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA',
    'Electricity(kWh)', 'SteamUse(kBtu)', 'NaturalGas(therms)', 'Electricity(kBtu)',
    'NaturalGas(kBtu)', 'TotalGHGEmissions', 'GHGEmissionsIntensity',
]
CIBLE = "SiteEnergyUse(kBtu)"

data = data.select([c for c in features_variables if c in data.columns])
print(len(data.columns), "colonnes retenues")

## 4. Colonnes avec plus de 70 % de valeurs manquantes

Sont comptés comme manquants : les `null`, les `NaN` (colonnes décimales) et les chaînes vides. Tous les comptages sont faits en **une seule agrégation**.

In [ ]:
def supprimer_colonnes_manquantes(df, colonnes, seuil=70.0):
    """Affiche le % de valeurs manquantes de chaque colonne, puis supprime celles qui dépassent le seuil."""
    colonnes = [c for c in colonnes if c in df.columns]
    exprs = [F.count(F.lit(1)).alias("__total__")]
    for c in colonnes:
        type_c = df.schema[c].dataType
        manquant = F.col(c).isNull()
        if isinstance(type_c, (DoubleType, FloatType)):
            manquant = manquant | F.isnan(F.col(c))
        elif isinstance(type_c, StringType):
            manquant = manquant | (F.trim(F.col(c)) == "")
        exprs.append(F.sum(F.when(manquant, 1).otherwise(0)).alias(c))

    ligne = df.select(exprs).first().asDict()
    total = ligne.pop("__total__")
    pct = {c: 100.0 * ligne[c] / total for c in colonnes}

    print(f"{'Colonne':<36}{'% manquant':>11}")
    print("-" * 47)
    for c, p in sorted(pct.items(), key=lambda x: -x[1]):
        print(f"{c:<36}{p:>10.2f}%{'   -> à supprimer' if p > seuil else ''}")

    a_supprimer = [c for c, p in pct.items() if p > seuil]
    print(f"\n{len(a_supprimer)} colonne(s) supprimée(s) : {a_supprimer}")
    return df.drop(*a_supprimer), a_supprimer


data, colonnes_supprimees = supprimer_colonnes_manquantes(data, features_variables, seuil=70)

## 5. Identifiants inutiles et nouvelle variable `Age`

`ZipCode` et `TaxParcelIdentificationNumber` sont des identifiants : ils n'apportent rien au modèle (le numéro de parcelle contient d'ailleurs des tirets, ex. `880490-0390`, et ne peut pas être converti en nombre).

L'âge du bâtiment est calculé au moment de la mesure : `Age = DataYear - YearBuilt`.

In [ ]:
data = data.drop("ZipCode", "TaxParcelIdentificationNumber")

data = (data
        .withColumn("Age", F.col("DataYear") - F.col("YearBuilt"))
        .drop("DataYear", "YearBuilt"))

data.select("Age").describe().show()

## 6. Nombres au format américain

Certaines colonnes numériques sont lues comme du texte car elles contiennent des séparateurs de milliers (`1,234,567.89`). On détecte ces colonnes automatiquement, on contrôle les valeurs non convertibles, puis on supprime les virgules et on convertit en `double`.

`try_cast` renvoie `null` au lieu de faire échouer tout le calcul sur une valeur invalide : avec le mode ANSI de Spark 4, un `cast` classique lève une erreur `CAST_INVALID_INPUT`.

In [ ]:
motif = r"^-?[0-9]{1,3}(,[0-9]{3})*(\.[0-9]+)?$|^-?[0-9]+(\.[0-9]+)?$"

vartocorect = []
for c in data.columns:
    if isinstance(data.schema[c].dataType, StringType):
        non_vides = data.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""))
        total = non_vides.count()
        conformes = non_vides.filter(F.trim(F.col(c)).rlike(motif)).count()
        if total > 0 and conformes / total >= 0.95:
            vartocorect.append(c)
print("Colonnes à convertir :", vartocorect)

# Contrôle : valeurs qui deviendraient null à la conversion
for c in vartocorect:
    n = data.filter(F.col(c).isNotNull() &
                    F.regexp_replace(F.col(c), ",", "").try_cast("double").isNull()).count()
    if n:
        print(f"{c} : {n} valeur(s) non convertible(s)")

for c in vartocorect:
    data = data.withColumn(c, F.regexp_replace(F.col(c), ",", "").try_cast("double"))

if vartocorect:
    data.select(vartocorect).show(3)

## 7. Filtrage : bâtiments non résidentiels

In [ ]:
target_buildings = ['NonResidential', 'School', 'University', 'Hospital']
data = data.filter(F.col("BuildingType").isin(target_buildings))

data.groupBy("BuildingType").count().orderBy(F.desc("count")).show()
print((data.count(), len(data.columns)))

## 8. Variable cible

On ne garde que les bâtiments dont la consommation est connue (`null` et `NaN` exclus), puis on examine ses valeurs extrêmes.

In [ ]:
data = data.filter(F.col(CIBLE).isNotNull() & ~F.isnan(F.col(CIBLE)))

data.select(CIBLE).describe().show()
print("Consommations <= 0 :", data.filter(F.col(CIBLE) <= 0).count())

## 9. Valeurs manquantes restantes

In [ ]:
def nb_manquants(c, type_c):
    cond = F.col(c).isNull()
    if isinstance(type_c, (DoubleType, FloatType)):
        cond = cond | F.isnan(F.col(c))
    return F.sum(F.when(cond, 1).otherwise(0)).alias(c)

total = data.count()
valeurs = data.select([nb_manquants(f.name, f.dataType) for f in data.schema.fields]).first().asDict()

resume = (spark.createDataFrame([(c, int(n), round(100.0 * n / total, 2)) for c, n in valeurs.items()],
                                ["colonne", "nb_null", "pct_null"])
          .filter(F.col("nb_null") > 0)
          .orderBy(F.desc("nb_null")))
resume.show(len(valeurs), truncate=False)

## 10. Variables numériques et catégorielles (lues dans le schéma)

In [ ]:
def listes_variables(df):
    num = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    cat = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    return num, cat

numerical_variables, categorical_variables = listes_variables(data)
print(len(numerical_variables), "numériques :", numerical_variables)
print(len(categorical_variables), "catégorielles :", categorical_variables)

## 11. Fuites de données (*data leakage*)

Ces colonnes sont des composantes ou des variantes de la consommation à prédire (électricité + gaz + vapeur = consommation totale ; EUI = consommation / surface ; émissions calculées à partir de la consommation). On ne les connaît qu'une fois la consommation mesurée : les garder reviendrait à donner la réponse au modèle.

In [ ]:
fuites = [
    'SiteEnergyUseWN(kBtu)', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)',
    'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)',
    'Electricity(kWh)', 'Electricity(kBtu)', 'NaturalGas(therms)', 'NaturalGas(kBtu)',
    'SteamUse(kBtu)', 'TotalGHGEmissions', 'GHGEmissionsIntensity',
]
data = data.drop(*fuites)
numerical_variables, categorical_variables = listes_variables(data)
print(numerical_variables)

## 12. Matrice de corrélation de Pearson (100 % Spark)

`F.corr` calcule chaque paire en ignorant les lignes où l'une des deux valeurs manque ; toutes les paires sont calculées en une seule passe.

In [ ]:
cols = numerical_variables
n = len(cols)
paires = [(i, j) for i in range(n) for j in range(i + 1, n)]

ligne = data.select([F.corr(cols[i], cols[j]).alias(f"{i}_{j}") for i, j in paires]).first()
corr = np.eye(n)
for i, j in paires:
    r = ligne[f"{i}_{j}"]
    corr[i, j] = corr[j, i] = np.nan if r is None else r

masque = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(np.ma.array(corr, mask=masque), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(cols, rotation=90, fontsize=8)
ax.set_yticks(range(n)); ax.set_yticklabels(cols, fontsize=8)
for i in range(n):
    for j in range(i + 1):
        if not np.isnan(corr[i, j]):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, shrink=0.6)
ax.set_title("Matrice de corrélation de Pearson")
plt.tight_layout(); plt.show()

In [ ]:
paires_df = (spark.createDataFrame(
                 [(cols[i], cols[j], float(corr[i, j])) for i, j in paires if not np.isnan(corr[i, j])],
                 ["var1", "var2", "r"])
             .withColumn("abs_r", F.abs("r")))

print("Corrélation avec la cible :")
(paires_df.filter((F.col("var1") == CIBLE) | (F.col("var2") == CIBLE))
          .withColumn("variable", F.when(F.col("var1") == CIBLE, F.col("var2")).otherwise(F.col("var1")))
          .select("variable", F.round("r", 3).alias("r"))
          .orderBy(F.desc(F.abs("r")))
          .show(n, truncate=False))

## 13. Suppression des variables redondantes (|r| > 0,7)

Pour chaque paire trop corrélée, on garde la première variable de la liste. Deux garde-fous : la cible n'est jamais supprimée, et l'on ne compare qu'aux variables conservées (sinon une suppression en entraîne d'autres sans raison).

In [ ]:
seuil = 0.70
columns_to_drop = set()
for i in range(len(cols)):
    if cols[i] == CIBLE:
        continue
    for j in range(i):
        if cols[j] in columns_to_drop:
            continue
        if not np.isnan(corr[i, j]) and abs(corr[i, j]) > seuil:
            columns_to_drop.add(cols[i])
            print(f"{cols[i]:<35} supprimée : r = {corr[i, j]:+.2f} avec {cols[j]}")
            break

data = data.drop(*columns_to_drop)
numerical_variables, categorical_variables = listes_variables(data)
print("\nVariables numériques restantes :", numerical_variables)

## 14. Encodage des variables catégorielles

* Normalisation du texte (majuscules, espaces) pour fusionner les variantes comme `Seattle` / `SEATTLE` ; les variables devenues constantes sont écartées.
* Moins de 40 modalités : `StringIndexer` puis `OneHotEncoder`.
* 40 modalités ou plus : `StringIndexer` seul (équivalent Spark du `LabelEncoder`), adapté aux modèles à base d'arbres.
* `handleInvalid="keep"` réserve un index aux valeurs manquantes et aux modalités inconnues.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# BuildingName est un nom propre (presque une valeur par bâtiment) : on l'écarte de l'encodage
categorical_variables = [c for c in categorical_variables if c != "BuildingName"]

for c in categorical_variables:
    data = data.withColumn(c, F.upper(F.trim(F.col(c))))

card = data.select([F.countDistinct(c).alias(c) for c in categorical_variables]).first().asDict()
print("Modalités :", card)

# Une variable à une seule modalité (ex. City = SEATTLE après normalisation) n'apporte aucune information
constantes = [c for c in categorical_variables if card[c] <= 1]
print("Constantes écartées :", constantes)
data = data.drop(*constantes)
categorical_variables = [c for c in categorical_variables if c not in constantes]

ohe_cols = [c for c in categorical_variables if card[c] < 40]
label_cols = [c for c in categorical_variables if card[c] >= 40]
print("One-hot       :", ohe_cols)
print("StringIndexer :", label_cols)

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in categorical_variables]
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in ohe_cols],
                        outputCols=[c + "_ohe" for c in ohe_cols], handleInvalid="keep")

modele_encodage = Pipeline(stages=indexers + [encoder]).fit(data)
data_encoded = modele_encodage.transform(data)

encoded_variables = [c + "_ohe" for c in ohe_cols] + [c + "_idx" for c in label_cols]
data_encoded.select(encoded_variables).show(5, truncate=False)

## 15. Sauvegarde au format Parquet

Le jeu préparé est sauvegardé en Parquet (format colonne, compressé, qui conserve les types et les vecteurs) pour l'étape de modélisation.

In [ ]:
variables_finales = [c for c in numerical_variables if c != CIBLE] + encoded_variables
print(len(variables_finales), "variables explicatives :", variables_finales)

data_encoded.write.mode("overwrite").parquet("building_energy_prepare.parquet")
print(data_encoded.count(), "lignes sauvegardées")

## Bilan

* Préparation entièrement réalisée avec Spark (aucun `toPandas()`), y compris la matrice de corrélation.
* Variables à fuite de données exclues avant l'analyse des corrélations.
* Jeu prêt pour la modélisation : `VectorAssembler` sur `variables_finales`, puis un modèle de régression (Random Forest ou Gradient Boosting, adaptés aux variables indexées).

**À noter :** en toute rigueur, l'encodage doit être ajusté (`fit`) sur le jeu d'entraînement seulement, après le `randomSplit`.